# 04 - Unsupervised NLP Topic Modelling

**Problem framing.** We have a corpus of documents and no labels. The goal is to discover the latent themes, attach readable labels to them, assign each document to a theme, and - crucially - judge whether the discovered topics are trustworthy. We use TF-IDF features with NMF and Latent Semantic Analysis (TruncatedSVD), and keep everything offline (no external APIs).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import NMF, TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

sys.path.append(str(Path.cwd() / "src"))

from unsup_lab.data import make_document_corpus
from unsup_lab.nlp import (
    assign_documents,
    clean_text,
    label_topics,
    topic_diversity,
    topic_term_table,
    umass_topic_coherence,
)

## Dataset

A small synthetic corpus with four latent themes (healthcare, finance, retail, IoT). The `hidden_topic` column is kept aside for *offline* evaluation only and never used during modelling.

In [ ]:
corpus = make_document_corpus(random_state=42)
print(corpus["hidden_topic"].value_counts().to_dict())
corpus.head()

## Text cleaning and vectorisation

`clean_text` lowercases and strips punctuation and digits. We then build TF-IDF features with unigrams and bigrams, dropping terms that appear in fewer than two documents.

In [ ]:
cleaned = corpus["text"].map(clean_text)
print("raw   :", corpus["text"].iloc[0][:80])
print("clean :", cleaned.iloc[0][:80])

vectorizer = TfidfVectorizer(min_df=2, max_df=0.9, ngram_range=(1, 2))
x_tfidf = vectorizer.fit_transform(cleaned)
terms = vectorizer.get_feature_names_out()
print("tf-idf matrix:", x_tfidf.shape)

## NMF topic model

In [ ]:
n_topics = 4
nmf = NMF(n_components=n_topics, init="nndsvda", random_state=42, max_iter=500)
document_topics = nmf.fit_transform(x_tfidf)

# Topic-term table and auto-generated labels from the top terms.
table = topic_term_table(nmf.components_, terms, n_terms=10)
labels = label_topics(nmf.components_, terms, n_terms=3)
labels

In [ ]:
table[table["topic"] == "topic_0"]

## Document-topic assignment

Each document is assigned to its dominant topic with a confidence (the topic's share of the row mass). The cross-tab against the hidden themes is an offline sanity check that the unsupervised topics line up with reality.

In [ ]:
assignment = assign_documents(document_topics)
assignment["topic_label"] = assignment["dominant_topic"].map(
    lambda i: labels[f"topic_{i}"]
)
assignment["hidden_topic"] = corpus["hidden_topic"].to_numpy()
print(f"mean assignment confidence: {assignment['confidence'].mean():.3f}")
pd.crosstab(assignment["dominant_topic"], assignment["hidden_topic"])

## Latent Semantic Analysis (TruncatedSVD)

LSA factorises the same TF-IDF matrix with a truncated SVD. Unlike NMF its components can be negative, so they read as contrasts between term groups rather than additive themes.

In [ ]:
svd = TruncatedSVD(n_components=4, random_state=42)
svd.fit(x_tfidf)
print("explained variance:", svd.explained_variance_ratio_.round(3))
topic_term_table(svd.components_, terms, n_terms=8)

## Topic quality diagnostics

Two complementary diagnostics:

- **UMass coherence** - do a topic's top terms actually co-occur in documents? Higher (less negative) is better.
- **Topic diversity** - the fraction of unique terms across topics; low diversity means topics overlap.

We also report NMF's reconstruction error (lower is a tighter fit).

In [ ]:
# A binary count matrix sharing the TF-IDF vocabulary lets us score coherence.
counter = CountVectorizer(vocabulary=vectorizer.vocabulary_, ngram_range=(1, 2))
counts = counter.fit_transform(cleaned)

top_terms = [
    group.sort_values("rank")["term"].tolist()
    for _, group in table.groupby("topic", sort=False)
]
coherence = umass_topic_coherence(top_terms, counts, vectorizer.vocabulary_)

diagnostics = pd.DataFrame(
    {
        "topic": list(labels.keys()),
        "label": list(labels.values()),
        "umass_coherence": np.round(coherence, 3),
    }
)
print(f"topic diversity: {topic_diversity(top_terms):.3f}")
print(f"NMF reconstruction error: {nmf.reconstruction_err_:.3f}")
diagnostics

## Failure modes

Topic models fail quietly. The cells below reproduce four common failure modes on this corpus so the symptoms are recognisable in real work.

### 1. Overlapping topics from over-factorising

Asking for more topics than the data contains splits coherent themes and repeats vocabulary, which shows up as a drop in topic diversity.

In [ ]:
for k in (4, 8, 12):
    model = NMF(n_components=k, init="nndsvda", random_state=42, max_iter=500)
    model.fit(x_tfidf)
    terms_k = [
        group.sort_values("rank")["term"].tolist()
        for _, group in topic_term_table(model.components_, terms, 10).groupby(
            "topic", sort=False
        )
    ]
    print(f"k={k:2d}  topic_diversity={topic_diversity(terms_k):.3f}")

### 2. Short documents

Truncating each document to a few tokens strips out most of the co-occurring signal terms NMF relies on. On this clean synthetic corpus the surviving tokens are so topic-specific that assignments stay confident, but each one now rests on just one or two terms - fragile, and easily flipped by noise on real-world text.

In [ ]:
short = cleaned.map(lambda text: " ".join(text.split()[:3]))
short_tfidf = vectorizer.transform(short)
full_terms = x_tfidf.getnnz(axis=1).mean()
short_terms = short_tfidf.getnnz(axis=1).mean()
print(f"mean signal terms per doc (full)   : {full_terms:.1f}")
print(f"mean signal terms per doc (3 words) : {short_terms:.1f}")

short_conf = assign_documents(nmf.transform(short_tfidf))["confidence"].mean()
print(f"3-word assignment confidence       : {short_conf:.3f} "
      "(still high, but built on very little)")

### 3. Vocabulary drift

Documents written with new vocabulary (here, ops/streaming jargon) fall outside the trained vocabulary, so their TF-IDF rows are nearly empty and the model cannot place them.

In [ ]:
drift_docs = [
    "telemetry payload ingestion throughput kafka broker",
    "kubernetes pod autoscaling latency sla observability",
]
drift_tfidf = vectorizer.transform([clean_text(doc) for doc in drift_docs])
nonzero_per_doc = drift_tfidf.getnnz(axis=1)
print("non-zero tf-idf terms per drifted doc:", nonzero_per_doc.tolist())
print("(0 means the document shares no vocabulary with the training corpus)")

### 4. Domain-specific terms

Raising `min_df` to suppress noise also discards rare but meaningful domain terms - a trade-off worth measuring rather than guessing.

In [ ]:
strict = TfidfVectorizer(min_df=8, max_df=0.9, ngram_range=(1, 2))
strict.fit(cleaned)
kept = set(strict.get_feature_names_out())
dropped = sorted(set(terms) - kept)
print(f"vocabulary: {len(terms)} terms at min_df=2 -> {len(kept)} at min_df=8")
print("examples of dropped terms:", dropped[:10])

## Interpretation and limitations

The NMF topics align cleanly with the four hidden themes and carry sensible auto-labels, with coherence and diversity confirming they are distinct. But the failure modes show how fragile this is: too many topics, short texts, drifting vocabulary, or aggressive frequency filtering each degrade the result silently.

**Limitations.**

- NMF and LSA are bag-of-words: they ignore word order and context, so synonyms and polysemy are invisible.
- The number of topics is a modelling choice; coherence and diversity guide it but do not determine it.
- Coherence here is computed on the same corpus that trained the model, so it measures internal consistency, not generalisation.
- Discovered topics are hypotheses for a domain expert to validate, not ground truth.